# 🧱 Part 20: Vision-Language Models: Connecting Images to Text

> **Previous context**: Text-only models read token sequences. VLMs add visual information to the same language-model pipeline.
> **Goal for this part**: Understand patch embeddings, vision encoders, projectors, Cross-Attention, and Flamingo-style gated fusion.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why VLMs need a bridge

Images are arrays of pixels; language models expect token-like vectors. A VLM must convert visual signals into representations the LLM can use.

## 1. Patch Embedding

An image can be split into patches, and each patch can become a vector, similar to a visual token.

## 2. Projector and fusion

A projector maps vision features into the language model dimension. Cross-Attention or gated blocks let text tokens read image features.

## 3. Training strategy

Many VLMs freeze some parts at first so visual ability grows without immediately destroying language ability.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Images become patch or vision features.
- [ ] A projector aligns visual features with the LLM hidden size.
- [ ] Cross-Attention lets language tokens read visual information.

Next, continue through the code cells for the Frontiers part and inspect the printed observations.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

torch.manual_seed(42)

### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
┌──┬──┬──┬──┐
Read the values printed above and connect them to the concept in this cell.├──┼──┼──┼──┤
Read the values printed above and connect them to the concept in this cell.├──┼──┼──┼──┤
│  │  │  │  │
└──┴──┴──┴──┘
```

Read the values printed above and connect them to the concept in this cell.

In [2]:
# Teaching note: follow this line to see the main step.
IMG_SIZE = 224
PATCH_SIZE = 16

# Teaching note: follow this line to see the main step.
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print(f"Read the values printed above and connect them to the concept in this cell.{fake_image.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.")

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE).unfold(2, PATCH_SIZE, PATCH_SIZE)
print(f"Read the values printed above and connect them to the concept in this cell.{patches.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.")

# Teaching note: follow this line to see the main step.
num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE  # 3×16×16 = 768
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print(f"Read the values printed above and connect them to the concept in this cell.{patches_flat.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.{num_patches}Read the values printed above and connect them to the concept in this cell.{patch_dim}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.{patch_dim}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.{patch_dim}Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

In [3]:
class PatchEmbedding(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.proj = nn.Conv2d(in_channels, d_model, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        '"""\n        Input: x shape = [batch, 3, 224, 224]\n        Output:   shape = [batch, num_patches, d_model]\n        """'
        x = self.proj(x)          # [batch, d_model, 14, 14]
        x = x.flatten(2)           # [batch, d_model, 196]
        x = x.transpose(1, 2)      # [batch, 196, d_model]
        return x

# Teaching note: follow this line to see the main step.
patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)  # Teaching note: follow this line to see the main step.
visual_tokens = patch_emb(dummy_img)

print(f"Read the values printed above and connect them to the concept in this cell.{dummy_img.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.{visual_tokens.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  ↑         ↑            ↑               ↑          ↑       ↑     ↑     ↑    ↑    ↑
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

In [4]:
# Teaching note: follow this line to see the main step.
d_model = 512
text_vocab_size = 1000

# Teaching note: follow this line to see the main step.
text_ids = torch.tensor([[5, 12, 78, 3, 90]])  # Teaching note: follow this line to see the main step.
text_emb = nn.Embedding(text_vocab_size, d_model)
text_vecs = text_emb(text_ids)  # [1, 5, 512]

# Teaching note: follow this line to see the main step.
visual_vecs = torch.randn(1, 196, d_model)  # Teaching note: follow this line to see the main step.

# Teaching note: follow this line to see the main step.
combined = torch.cat([visual_vecs, text_vecs], dim=1)

print(f"Read the values printed above and connect them to the concept in this cell.{visual_vecs.shape}")
print(f"Text token: {text_vecs.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.{combined.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

Read the values printed above and connect them to the concept in this cell.Text token: torch.Size([1, 5, 512])Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.                         │
                         ▼
              ┌─────────────────────┐
Read the values printed above and connect them to the concept in this cell.              │   (Conv2d 16×16)    │
              └─────────┬───────────┘
                        │ [196, d_model]
                        ▼
              ┌─────────────────────┐
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.              └─────────┬───────────┘
                        │
       ┌────────────────┼────────────────┐
       │                │                │
       ▼                ▼                ▼
Read the values printed above and connect them to the concept in this cell.       │                │                │         │           │
       └────────────────┴────────────────┴─────────┴───────────┘
                              │
                              ▼
                    ┌─────────────────┐
Read the values printed above and connect them to the concept in this cell.                    │  (Transformer)  │
                    └────────┬────────┘
                             │
                             ▼
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Conclusion
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Cross-Attn ✅ need 0 Flamingo / OpenFlamingoRead the values printed above and connect them to the concept in this cell.```

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Exercise passed: you have understood this step.
```
Read the values printed above and connect them to the concept in this cell.  Layer 1: Self-Attn → Cross-Attn → FFN → hidden₁
  Layer 2: Self-Attn → Cross-Attn → FFN → hidden₂  
  Layer 3: Self-Attn → Cross-Attn → FFN → hidden₃
  ...
                    ↑              ↑              ↑
                     \             |             /
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
```
Self-Attention:
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Cross-Attention:
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
$$\text{CrossAttn}(X_{text}, X_{vis}) = \text{softmax}\left(\frac{Q_{text} K_{vis}^T}{\sqrt{d_k}}\right) V_{vis}$$

Among them:- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
$$X_{out} = X_{in} + \tanh(\alpha) \cdot \text{CrossAttn}(X_{in}, X_{vis})$$

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.

# Read the values printed above and connect them to the concept in this cell.# Read the values printed above and connect them to the concept in this cell.
class CrossAttention(nn.Module):
    """
Read the values printed above and connect them to the concept in this cell.    
Read the values printed above and connect them to the concept in this cell.    - Read the values printed above and connect them to the concept in this cell.    - Read the values printed above and connect them to the concept in this cell.    """
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5
        
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, text_hidden, image_features):
        """
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.        """
        B, T, D = text_hidden.shape
        N = image_features.shape[1]
        
        # Read the values printed above and connect them to the concept in this cell.        Q = self.q_proj(text_hidden)      # [B, T, D]
        K = self.k_proj(image_features)   # [B, N, D]
        V = self.v_proj(image_features)   # [B, N, D]
        
        # Multi-head reshape        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, T, d]
        K = K.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d]
        V = V.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d]
        
        # Scaled Dot-Product Attention        # Q @ K^T: [B, h, T, d] @ [B, h, d, N] → [B, h, T, N]        # Read the values printed above and connect them to the concept in this cell.        attn_weights = (Q @ K.transpose(-2, -1)) * self.scale  # [B, h, T, N]
        attn_weights = F.softmax(attn_weights, dim=-1)
        
        # Read the values printed above and connect them to the concept in this cell.        # Read the values printed above and connect them to the concept in this cell.        out = attn_weights @ V  # [B, h, T, d]
        out = out.transpose(1, 2).reshape(B, T, D)  # [B, T, D]
        return self.out_proj(out)


class FlamingoGatedCrossAttnBlock(nn.Module):
    """
Read the values printed above and connect them to the concept in this cell.    Self-Attention → Gated Cross-Attention → FFN
    
Read the values printed above and connect them to the concept in this cell.    """
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        # Read the values printed above and connect them to the concept in this cell.        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, batch_first=True
        )
        # Read the values printed above and connect them to the concept in this cell.        self.cross_attn = CrossAttention(d_model, n_heads)
        # FFN        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        # Layer Norms        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        # Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
    def forward(self, x, image_features, causal_mask=None):
        """
x: [B, T, D] Text hidden statesRead the values printed above and connect them to the concept in this cell.        """
        # Read the values printed above and connect them to the concept in this cell.        x = x + self.self_attn(x, x, x, attn_mask=causal_mask)[0]
        x = self.norm1(x)
        
        # Read the values printed above and connect them to the concept in this cell.        #    Read the values printed above and connect them to the concept in this cell.        #    Read the values printed above and connect them to the concept in this cell.        gate = torch.tanh(self.alpha)
        x = x + gate * self.cross_attn(x, image_features)
        x = self.norm2(x)
        
        # 3. FFN        x = x + self.ffn(x)
        x = self.norm3(x)
        return x


# Read the values printed above and connect them to the concept in this cell.class LLaVABlock(nn.Module):
Read the values printed above and connect them to the concept in this cell.    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, causal_mask=None):
        x = x + self.self_attn(x, x, x, attn_mask=causal_mask)[0]
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x


# Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
d_model, n_heads = 512, 8

# Read the values printed above and connect them to the concept in this cell.batch = 1
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.text_hidden = torch.randn(batch, text_len, d_model)
image_feat = torch.randn(batch, num_vis, d_model)

# Read the values printed above and connect them to the concept in this cell.combined = torch.cat([image_feat, text_hidden], dim=1)  # [1, 196+50, 512]
llava_block = LLaVABlock(d_model, n_heads)
llava_out = llava_block(combined)
print(f"LLaVA (Visual Token):")
print(f" Input: [{batch}, {text_len + num_vis}, {d_model}]")Read the values printed above and connect them to the concept in this cell.print(f"  FLOPs ∝ (196+50)² = {246**2:,}")

# Read the values printed above and connect them to the concept in this cell.flamingo_block = FlamingoGatedCrossAttnBlock(d_model, n_heads)
with torch.no_grad():
    flamingo_out = flamingo_block(text_hidden, image_feat)
print(f"\nFlamingo (Cross-Attention):")
print(f" Input: [{batch}, {text_len}, {d_model}]")Read the values printed above and connect them to the concept in this cell.print(f"  Cross-Attention: Q({text_len}) @ K({num_vis})")
print(f"  FLOPs ∝ 50² + 50×196 = {50**2 + 50*196:,}")
print(f"  Gate α = {flamingo_block.alpha.item():.4f} → tanh(α) = {torch.tanh(flamingo_block.alpha).item():.4f}")

ConclusionRead the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Reason- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.

# Number of tokens: for patch_size in [8, 14, 16, 32]:
    num_patches = (224 // patch_size) ** 2
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.
         Vision Encoder ──→ Projector ──→ LLM
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.LossRead the values printed above and connect them to the concept in this cell.

Read the values printed above and connect them to the concept in this cell.
         Vision Encoder ──→ Projector ──→ LLM
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.LossRead the values printed above and connect them to the concept in this cell.```

#### Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.└── 7B Model ≈ 14GB(bf16) + 14GB(grad) + 28GB(optim) ≈ 56GB
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.└── 7B Model ≈ 14GB(bf16) + 0.5GB(LoRA) + 1GB(optim) ≈ 16GBRead the values printed above and connect them to the concept in this cell.```

```python
# Read the values printed above and connect them to the concept in this cell.def configure_training(vision_encoder, projector, llm, stage=1):
Read the values printed above and connect them to the concept in this cell.    # Read the values printed above and connect them to the concept in this cell.    for p in vision_encoder.parameters():
        p.requires_grad = False
    
    # Read the values printed above and connect them to the concept in this cell.    for p in projector.parameters():
        p.requires_grad = True
    
    # Read the values printed above and connect them to the concept in this cell.    if stage == 1:
        # Read the values printed above and connect them to the concept in this cell.        for p in llm.parameters():
            p.requires_grad = False
Read the values printed above and connect them to the concept in this cell.    elif stage == 2:
        # Read the values printed above and connect them to the concept in this cell.        for p in llm.parameters():
            p.requires_grad = True
        total = (sum(p.numel() for p in projector.parameters()) + 
                 sum(p.numel() for p in llm.parameters()))
Read the values printed above and connect them to the concept in this cell.    else:
        raise ValueError(f"Unknown stage: {stage}")

# Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.print()
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

#### Read the values printed above and connect them to the concept in this cell.
|Model| Vision Encoder | Projector | LLM |Read the values printed above and connect them to the concept in this cell.|
|------|:---:|:---:|:---:|------|
| **LLaVA 1.5** |Read the values printed above and connect them to the concept in this cell.|🔥 training| S1 ❄️, S2 🔥 |Read the values printed above and connect them to the concept in this cell.|
| **LLaVA 1.6** |Read the values printed above and connect them to the concept in this cell.|🔥 training| S1 ❄️, S2 🔥 |Read the values printed above and connect them to the concept in this cell.|
| **Flamingo** |Read the values printed above and connect them to the concept in this cell.|🔥 training(cross-attn)|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
| **BLIP-2** |Read the values printed above and connect them to the concept in this cell.|🔥 training(Q-Former)|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
| **mPLUG-Owl** |Read the values printed above and connect them to the concept in this cell.|🔥 training|🔥 training(LoRA)|Read the values printed above and connect them to the concept in this cell.|
| **CogVLM** |Read the values printed above and connect them to the concept in this cell.|🔥 training|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|

Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.│
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.│
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.│
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
LLaVA v1:  Linear(d_vis → d_llm)
Read the values printed above and connect them to the concept in this cell.
LLaVA 1.5: MLP(d_vis → d_llm)  + 2-layer GELU
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```python
# Read the values printed above and connect them to the concept in this cell.class LLaVAProjector(nn.Module):
Read the values printed above and connect them to the concept in this cell.    def __init__(self, vis_dim=1024, llm_dim=4096, hidden_multiplier=2):
        super().__init__()
        hidden_dim = vis_dim * hidden_multiplier
        self.mlp = nn.Sequential(
            nn.Linear(vis_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, llm_dim),
        )
    
    def forward(self, visual_features):
        """visual_features: [B, N, vis_dim] → [B, N, llm_dim]"""
        return self.mlp(visual_features)
```

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.     ↑                                                    ↑
Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
|Model|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
|------|------|------|
| **LLaVA** |Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
| **Qwen-VL** | `<img>...</img>` |Read the values printed above and connect them to the concept in this cell.|
| **InternVL** | `<IMG_CONTEXT>` |Read the values printed above and connect them to the concept in this cell.|
| **Gemini** |Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|

Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.  token: [vis₀, vis₁, ..., vis₁₉₅, text₀, text₁, text₂, text₃, text₄]
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.```
Read the values printed above and connect them to the concept in this cell.    196 token            196 token          196 token              196 token
```
Read the values printed above and connect them to the concept in this cell.
#### Read the values printed above and connect them to the concept in this cell.
```python
# Read the values printed above and connect them to the concept in this cell.vision_output = clip_model.vision_model(pixel_values)

# Read the values printed above and connect them to the concept in this cell.features = vision_output.last_hidden_state  # [B, 196, 1024]

# Read the values printed above and connect them to the concept in this cell.# Read the values printed above and connect them to the concept in this cell.features = vision_output.hidden_states[-2]  # [B, 196, 1024]

# Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

### Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

---

## Read the values printed above and connect them to the concept in this cell.
### Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.5. Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.6. Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.   - Read the values printed above and connect them to the concept in this cell.7. Read the values printed above and connect them to the concept in this cell.8. Read the values printed above and connect them to the concept in this cell.
### Quick reference
|Read the values printed above and connect them to the concept in this cell.|One-line explanation|
|------|-----------|
| **Patch Embedding** |Read the values printed above and connect them to the concept in this cell.|
| **Projector** |Read the values printed above and connect them to the concept in this cell.|
| **Cross-Attention** |Read the values printed above and connect them to the concept in this cell.|
| **Tanh Gating** |Read the values printed above and connect them to the concept in this cell.|
|Read the values printed above and connect them to the concept in this cell.|Read the values printed above and connect them to the concept in this cell.|
| **AnyRes** |Read the values printed above and connect them to the concept in this cell.|
| **Q-Former** |Read the values printed above and connect them to the concept in this cell.|
| **LoRA for VLM** |Read the values printed above and connect them to the concept in this cell.|
| **Special Tokens** |Read the values printed above and connect them to the concept in this cell.|

### Read the values printed above and connect them to the concept in this cell.
```
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.│
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.│
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.│
Read the values printed above and connect them to the concept in this cell.Read the values printed above and connect them to the concept in this cell.```

Read the values printed above and connect them to the concept in this cell.

In [5]:
class MiniVLM(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, text_vocab_size=100, d_model=128, num_heads=2, num_layers=2):
        super().__init__()
        self.vision = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, d_model=d_model)
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, text_vocab_size)

    def forward(self, images, text_ids):
        visual_tokens = self.vision(images)
        text_tokens = self.text_embedding(text_ids)
        x = torch.cat([visual_tokens, text_tokens], dim=1)
        x = self.layers(x)
        x = self.norm(x)
        return self.lm_head(x)

# Teaching note: follow this line to see the main step.
vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

# Teaching note: follow this line to see the main step.
dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))  # Teaching note: follow this line to see the main step.

logits = vlm(dummy_images, dummy_text)

print(f"Read the values printed above and connect them to the concept in this cell.{dummy_images.shape}] + Text [{dummy_text.shape}]")
print(f"Output: {logits.shape}")
print(f"Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.{sum(p.numel() for p in vlm.parameters()):,}")

Read the values printed above and connect them to the concept in this cell.Output: torch.Size([1, 206, 100])Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

---

## Read the values printed above and connect them to the concept in this cell.
1. Read the values printed above and connect them to the concept in this cell.2. Read the values printed above and connect them to the concept in this cell.3. Read the values printed above and connect them to the concept in this cell.4. Read the values printed above and connect them to the concept in this cell.5. Read the values printed above and connect them to the concept in this cell.6. Read the values printed above and connect them to the concept in this cell.7. Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.